# Notebook 07 of 7 — Offline Recording + End-to-End (Track B: Free-only)

*Portfolio Intelligence Engine — User Guide Series, **Track B (free sources only)**.*
[Series README](../portfolio/README.md) · [Story Bible](../portfolio/STORY_BIBLE.md) · Epic [#1428](https://github.com/prajoria/OpenBB/issues/1428) · This notebook [#1443](https://github.com/prajoria/OpenBB/issues/1443) · Track A counterpart: [`../portfolio/07-offline-recording-and-end-to-end.ipynb`](../portfolio/07-offline-recording-and-end-to-end.ipynb).

---

## Where we are in Sam's story

Six notebooks in. NB01-NB06 (Track B) each pickled their evidence into `.notebook_state/` under `_free.pkl` names. This capstone is the reunion: reload every Track B artifact and compose one self-contained HTML report — Sam's Monday-morning read — sourced end-to-end from free-authoritative providers (CBOE + SEC + yfinance-recorded).

The Track A capstone answered *does the paid pipeline hang together?* Track B answers the harder question:

> *Can Sam ship the same report without paying for FMP?*

Answer: mostly yes — the Analysis composite (`Phase7Result`) requires `fmp_cached` and is honestly deferred to Track A, the same handoff NB02 (Track B) §7 already documented. Everything else lands.


### Provider chain for this notebook (Track B / [#1443](https://github.com/prajoria/OpenBB/issues/1443))

| Section | Data source |
|---|---|
| §1 scrape-record CLI probe | local (optional freshness check) |
| §2 end-to-end reunion — HTML report | pickled `_free.pkl` artifacts on disk |
| §3 timed run — P2 evidence + P3 x-ray + P6 backtest | `sec` + `cboe` |

**Optionality vs Track A.** In Track A the scrape-record snapshots are the primary evidence for insider/institutional flows. In Track B those flows are already covered by SEC N-PORT (NB03) and SEC insiders (NB04) — so §1 here is a *freshness spot-check*, not the primary source of truth.


## 0. Before we run anything

Same venv rule as every notebook in the series — `.venv_portfolio`. State goes into `.notebook_state/` (gitignored). We write ONLY to `portfolio_report_free.html`; Track A's `portfolio_report.html` is left alone so the reader can diff the two.


In [ ]:
# [Track B / NB07 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio, not the current interpreter.\n"
    "See NB01 §0 for setup.\n"
    f"Currently running: {sys.executable}"
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"Interpreter:              {pathlib.Path(sys.executable).name}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
Interpreter:              python.exe
venv sanity check:        passed (interpreter contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


## 0.5 Why free-only NB07 looks different from Track A

Track A NB07 leans on `scrape-record` snapshots as a primary evidence path — the widget-scraper CLI captures Fidelity/Robinhood-style positional flows that FMP's institutional endpoints don't cover. Track B doesn't need that lane because SEC N-PORT (quarterly, authoritative for 40-Act funds) plus SEC insider filings (Forms 3/4/5, 8-K) already cover the same beats — for free, from the primary regulatory source. So §1 here reduces `scrape-record` to a freshness spot-check rather than a load-bearing evidence path.

The reunion in §2 is the same six-section HTML report as Track A, sourced from the `_free.pkl` artifacts. Two intentional differences:

1. **§2 Deep dive** shows the P2 free-evidence bundle (SEC filings +    CBOE quote + insider transactions) and links Sam back to Track A    NB02 for the composite score — the same honest hand-off NB02    Track B §7 already documented.
2. **§6 Backtest** reports CBOE-sourced metrics (price-only) rather    than FMP total-return-adjusted, so Sharpe / CAGR run ~1-2%/yr    lower on dividend-paying names (BND, VNQ, GLD). NB06 §0.5 has the    fine print.

Bare-term pointers (all cited with Investopedia links in Track A NB07; not re-cited here): backtesting, look-ahead bias, drawdown, Sharpe ratio, HHI, effective N, look-through, insider transactions, 13F, N-PORT.


## 1. `scrape-record` freshness spot-check (optional)

Track A treated `scrape-record list` + `verify` as a load-bearing step because the paid path's institutional-flow evidence lives in those snapshots. In Track B the primary evidence lives in SEC N-PORT (NB03) and SEC insiders (NB04), so this cell is optional freshness telemetry: if the CLI is installed and any snapshots exist, spot-verify they still round-trip; otherwise, skip cleanly.


In [ ]:
# [Track B / NB07 §1] scrape-record CLI probe — freshness spot-check (optional)
# Shell out to the CLI; skip cleanly if unavailable — Track B does
# NOT depend on scrape-record snapshots (SEC N-PORT + SEC insiders
# already cover the same evidence beats).
import subprocess, sys, shutil
from collections import Counter
from pathlib import Path

def _cli() -> list[str]:
    exe_dir = Path(sys.executable).parent
    for name in ("scrape-record.exe", "scrape-record"):
        candidate = exe_dir / name
        if candidate.exists():
            return [str(candidate)]
    found = shutil.which("scrape-record")
    if found:
        return [found]
    return [sys.executable, "-c", "from scrape_record.cli import main; main()"]

cli = _cli()
try:
    proc = subprocess.run(cli + ["list"], capture_output=True, text=True, timeout=30)
except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
    print(f"scrape-record CLI unavailable ({type(exc).__name__}: {exc}).")
    print("  Track B skip: SEC N-PORT (NB03) + SEC insiders (NB04) cover the same beats.")
else:
    if proc.returncode != 0:
        print(f"scrape-record list exited {proc.returncode}")
        tail = (proc.stderr or proc.stdout)[:240]
        print(f"  {tail}")
        print("  Track B skip: this is not a load-bearing dependency.")
    else:
        lines = [ln for ln in proc.stdout.splitlines() if ln.strip()]
        counts: Counter[str] = Counter()
        for ln in lines[1:]:
            parts = ln.split()
            if len(parts) >= 3:
                counts[parts[0]] += 1
        total = sum(counts.values())
        print(f"Snapshots on disk: {total} across {len(counts)} recording type(s)")
        if total == 0:
            print("  No snapshots recorded yet — Track B does not require them.")
        else:
            print()
            print(f"  {'Recording':<40} {'Count':>6}")
            for name, n in counts.most_common():
                print(f"  {name:<40} {n:>6}")
            print()
            print("Freshness spot-check (one snapshot per recording type):")
            drift = 0
            for rec_name in counts:
                symbol = None
                for ln in lines[1:]:
                    parts = ln.split()
                    if parts and parts[0] == rec_name:
                        symbol = parts[1]
                        break
                if not symbol:
                    continue
                try:
                    vp = subprocess.run(
                        cli + ["verify", rec_name, "--symbol", symbol],
                        capture_output=True, text=True, timeout=30,
                    )
                    ok = vp.returncode == 0
                    marker = "ok" if ok else "DRIFT"
                    if not ok:
                        drift += 1
                    print(f"  {rec_name:<40} {symbol:<8}  {marker}")
                except subprocess.TimeoutExpired:
                    print(f"  {rec_name:<40} {symbol:<8}  TIMEOUT")
                    drift += 1
            if drift == 0:
                print("  No drift detected on Track B optional snapshots.")
            else:
                print(f"  {drift} recording(s) drifted — Track B is unaffected.")


Snapshots on disk: 0 across 0 recording type(s)
  No snapshots recorded yet — Track B does not require them.


## 2. End-to-end reunion — compose the Track B HTML report

Load every `_free.pkl` artifact NB01-NB06 (Track B) dropped in `.notebook_state/`, compose one self-contained HTML doc, and write to `portfolio_report_free.html`. Same six-section shape as Track A NB07 §5:

1. **Basket** — `basket.json` (shared with Track A, unchanged).
2. **MSFT Deep Dive** — `msft_free_evidence.pkl` (SEC filings + CBOE    quote + insider transactions); composite score deferred to Track A.
3. **Basket X-Ray** — `xray_free.pkl` (N-PORT look-through, HHI,    effective-N, top sectors).
4. **Events & Smart Money** — `smart_money_free.pkl` (SEC insider    conviction + `events_free.pkl`).
5. **What-If & Paper** — `paper_blotter_free.pkl` (CBOE-quoted paper    blotter + Brinson attribution).
6. **Backtest & Validation** — `backtest_result_free.pkl` (CBOE    metrics + sweep best + PBO verdict).

Missing artifacts trigger a WARNING + section skip — never a crash. Pickle safety: trusted-local-only per repo policy.


In [ ]:
# [Track B / NB07 §2] End-to-end capstone — reload _free artifacts + compose HTML
# Reads every NB01-NB06 (Track B) pickle/json from .notebook_state/ and
# writes a single self-contained HTML file to portfolio_report_free.html.
# Missing artifacts trigger a WARNING and section skip — never a crash.
# Pickle safety: trusted-local-only per repo policy.
import json
import pickle  # noqa: S403  # trusted local; .notebook_state/ is gitignored
import html
from pathlib import Path

STATE = Path(".notebook_state")
OUT = STATE / "portfolio_report_free.html"

def _load_pickle(name: str):
    p = STATE / name
    if not p.exists():
        print(f"WARNING: {name} missing — skipping section")
        return None
    try:
        return pickle.loads(p.read_bytes())  # noqa: S301
    except Exception as exc:  # noqa: BLE001
        print(f"WARNING: {name} unpickle failed ({exc}) — skipping section")
        return None

def _load_json(name: str):
    p = STATE / name
    if not p.exists():
        print(f"WARNING: {name} missing — skipping section")
        return None
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except Exception as exc:  # noqa: BLE001
        print(f"WARNING: {name} parse failed ({exc}) — skipping section")
        return None

basket = _load_json("basket.json")
msft_evidence = _load_pickle("msft_free_evidence.pkl")
xray = _load_pickle("xray_free.pkl")
smart_money = _load_pickle("smart_money_free.pkl")
paper = _load_pickle("paper_blotter_free.pkl")
backtest = _load_pickle("backtest_result_free.pkl")

loaded = {
    "basket.json":              basket is not None,
    "msft_free_evidence.pkl":   msft_evidence is not None,
    "xray_free.pkl":            xray is not None,
    "smart_money_free.pkl":     smart_money is not None,
    "paper_blotter_free.pkl":   paper is not None,
    "backtest_result_free.pkl": backtest is not None,
}
print()
print("Artifact load summary:")
for name, ok in loaded.items():
    print(f"  {name:<32} {'ok' if ok else 'MISSING'}")

def esc(v) -> str:
    return html.escape(str(v))

CSS = """
body { font-family: -apple-system, Segoe UI, Roboto, sans-serif; max-width: 960px;
       margin: 2em auto; padding: 0 1em; color: #222; line-height: 1.5; }
h1 { border-bottom: 3px solid #2f855a; padding-bottom: .3em; }
h2 { color: #2f855a; margin-top: 2em; border-bottom: 1px solid #cbd5e0;
     padding-bottom: .2em; }
.skip { color: #a0522d; font-style: italic; }
.handoff { color: #2b6cb0; font-style: italic; }
table { border-collapse: collapse; margin: 1em 0; }
th, td { border: 1px solid #cbd5e0; padding: .35em .7em; text-align: left; }
th { background: #f0fff4; }
.metric { display: inline-block; margin-right: 1.5em; }
.metric .k { color: #4a5568; font-size: .85em; }
.metric .v { font-weight: bold; font-size: 1.1em; }
.warn { color: #b7410e; }
.footer { color: #718096; font-size: .85em; margin-top: 3em;
          border-top: 1px solid #cbd5e0; padding-top: 1em; }
.badge { display: inline-block; background: #f0fff4; color: #2f855a;
         border: 1px solid #9ae6b4; padding: .1em .5em; border-radius: 3px;
         font-size: .8em; }
"""

parts: list[str] = []
parts.append("<!doctype html><html><head><meta charset='utf-8'>")
parts.append("<title>Portfolio Intelligence Report (Track B — Free)</title>")
parts.append(f"<style>{CSS}</style></head><body>")
parts.append("<h1>Portfolio Intelligence — Monday-Morning Report "
             "<span class='badge'>Track B · free-only</span></h1>")
parts.append("<p><em>Composed from NB01-NB06 (Track B) artifacts in <code>.notebook_state/</code>. "
             "Providers: SEC + CBOE + yfinance-recorded. No paid FMP calls.</em></p>")

# --- 1. Basket ---
parts.append("<h2>1. Basket (NB01/NB03)</h2>")
if basket:
    rows = sorted(basket, key=lambda r: -r.get("weight", 0))[:12]
    parts.append("<table><tr><th>Symbol</th><th>Weight</th><th>Kind</th><th>Note</th></tr>")
    for r in rows:
        sym = r.get("symbol") or r.get("ticker", "?")
        parts.append(
            f"<tr><td>{esc(sym)}</td><td>{r.get('weight', 0)*100:.1f}%</td>"
            f"<td>{esc(r.get('kind',''))}</td><td>{esc(r.get('note',''))}</td></tr>"
        )
    parts.append("</table>")
else:
    parts.append("<p class='skip'>basket.json missing — section skipped.</p>")

# --- 2. Deep dive (free evidence + honest hand-off) ---
parts.append("<h2>2. MSFT Deep Dive (NB02, free evidence)</h2>")
if msft_evidence:
    sym = msft_evidence.get("symbol", "?")
    chain = msft_evidence.get("provider_chain", [])
    n_filings = len(msft_evidence.get("p1_filings", []) or [])
    n_income = len(msft_evidence.get("p2_income", []) or [])
    n_insiders = len(msft_evidence.get("p6_insiders", []) or [])
    quote = msft_evidence.get("p4_quote", {}) or {}
    parts.append(f"<p><span class='metric'><span class='k'>Symbol</span><br>"
                 f"<span class='v'>{esc(sym)}</span></span>"
                 f"<span class='metric'><span class='k'>Filings (P1)</span><br>"
                 f"<span class='v'>{n_filings}</span></span>"
                 f"<span class='metric'><span class='k'>Income rows (P2)</span><br>"
                 f"<span class='v'>{n_income}</span></span>"
                 f"<span class='metric'><span class='k'>Insider tx (P6)</span><br>"
                 f"<span class='v'>{n_insiders}</span></span></p>")
    bid = quote.get("bid")
    ask = quote.get("ask")
    if bid is not None and ask is not None:
        parts.append(f"<p><strong>P4 quote (CBOE):</strong> bid ${bid:.2f} / ask ${ask:.2f}</p>")
    parts.append(f"<p><strong>Provider chain:</strong> {esc(' → '.join(chain))}</p>")
    p7 = msft_evidence.get("p7_composite", {}) or {}
    note = p7.get("note", "")
    ref = p7.get("track_a_reference", "")
    if note or ref:
        parts.append(f"<p class='handoff'><strong>P7 composite score:</strong> "
                     f"{esc(note)} {esc(ref)}</p>")
else:
    parts.append("<p class='skip'>msft_free_evidence.pkl missing — section skipped.</p>")

# --- 3. X-Ray ---
parts.append("<h2>3. Basket X-Ray (NB03)</h2>")
if xray:
    eff = xray.get("effective_positions", {}) or {}
    sectors = xray.get("xray_by_sector", {}) or xray.get("naive_by_sector", {}) or {}
    hhi = xray.get("hhi_xray") or xray.get("hhi_raw")
    neff = xray.get("neff_xray") or xray.get("neff_raw")
    hhi_s = f"{hhi:.4f}" if isinstance(hhi, (int, float)) else esc(hhi)
    neff_s = f"{neff:.2f}" if isinstance(neff, (int, float)) else esc(neff)
    parts.append(f"<p><span class='metric'><span class='k'>Effective positions</span><br>"
                 f"<span class='v'>{len(eff)}</span></span>"
                 f"<span class='metric'><span class='k'>HHI</span><br>"
                 f"<span class='v'>{hhi_s}</span></span>"
                 f"<span class='metric'><span class='k'>Effective-N</span><br>"
                 f"<span class='v'>{neff_s}</span></span></p>")
    if sectors:
        top_sectors = sorted(sectors.items(), key=lambda kv: -kv[1])[:3]
        parts.append("<p><strong>Top-3 sectors:</strong></p>"
                     "<table><tr><th>Sector</th><th>Weight</th></tr>")
        for name, w in top_sectors:
            parts.append(f"<tr><td>{esc(name)}</td><td>{w*100:.2f}%</td></tr>")
        parts.append("</table>")
else:
    parts.append("<p class='skip'>xray_free.pkl missing — section skipped.</p>")

# --- 4. Events & Smart Money ---
parts.append("<h2>4. Events &amp; Smart Money (NB04)</h2>")
if smart_money:
    top = smart_money.get("top_conviction", []) or []
    warns = smart_money.get("warnings", []) or []
    if top:
        parts.append("<table><tr><th>Symbol</th><th>Composite</th><th>Signals</th></tr>")
        for it in top[:5]:
            sym = it.get("symbol") if isinstance(it, dict) else getattr(it, "symbol", "?")
            comp = it.get("composite", 0.0) if isinstance(it, dict) else getattr(it, "composite", 0.0)
            n = it.get("signal_count", 0) if isinstance(it, dict) else getattr(it, "signal_count", 0)
            comp_s = f"{comp:+.2f}" if isinstance(comp, (int, float)) else esc(comp)
            parts.append(f"<tr><td>{esc(sym)}</td><td>{comp_s}</td><td>{esc(n)}</td></tr>")
        parts.append("</table>")
    else:
        parts.append("<p><em>No smart-money conviction rows (upstream SEC feed "
                     "returned no ranked candidates on this run).</em></p>")
    if warns:
        parts.append("<p class='warn'><strong>Warnings:</strong></p><ul>")
        for w in warns:
            parts.append(f"<li>{esc(w)}</li>")
        parts.append("</ul>")
else:
    parts.append("<p class='skip'>smart_money_free.pkl missing — section skipped.</p>")

# --- 5. What-If & Paper ---
parts.append("<h2>5. What-If &amp; Paper Blotter (NB05)</h2>")
if paper:
    trades = paper.get("trades_submitted", []) or []
    wf = paper.get("whatif_diff", {}) or {}
    end_cash = paper.get("ending_cash")
    start_cash = paper.get("starting_cash")
    end_s = f"${end_cash:,.2f}" if isinstance(end_cash, (int, float)) else esc(end_cash)
    start_s = f"${start_cash:,.2f}" if isinstance(start_cash, (int, float)) else esc(start_cash)
    hhi_d = wf.get('hhi_delta', 0)
    hhi_d_s = f"{hhi_d:+.4f}" if isinstance(hhi_d, (int, float)) else esc(hhi_d)
    parts.append(f"<p><span class='metric'><span class='k'>Trades submitted</span><br>"
                 f"<span class='v'>{len(trades)}</span></span>"
                 f"<span class='metric'><span class='k'>HHI Δ</span><br>"
                 f"<span class='v'>{hhi_d_s}</span></span>"
                 f"<span class='metric'><span class='k'>Ending cash</span><br>"
                 f"<span class='v'>{end_s}</span></span>"
                 f"<span class='metric'><span class='k'>Starting cash</span><br>"
                 f"<span class='v'>{start_s}</span></span></p>")
    if trades:
        parts.append("<table><tr><th>Symbol</th><th>Action</th>"
                     "<th>ΔShares</th><th>Rationale</th></tr>")
        for t in trades:
            row = t if isinstance(t, dict) else {}
            parts.append(
                f"<tr><td>{esc(row.get('symbol','?'))}</td>"
                f"<td>{esc(row.get('action',''))}</td>"
                f"<td>{esc(row.get('delta_shares',''))}</td>"
                f"<td>{esc(row.get('rationale',''))}</td></tr>"
            )
        parts.append("</table>")
    qs = paper.get("quote_source")
    if qs:
        parts.append(f"<p><strong>Quote source:</strong> {esc(qs)}</p>")
else:
    parts.append("<p class='skip'>paper_blotter_free.pkl missing — section skipped.</p>")

# --- 6. Backtest & Validation ---
parts.append("<h2>6. Backtest &amp; Validation (NB06)</h2>")
if backtest:
    m = backtest.get("metrics", {}) or {}
    sweep = backtest.get("sweep_best", {}) or {}
    val = backtest.get("validation", {}) or {}
    provider = backtest.get("provider", "?")
    pbo = val.get("pbo")
    if pbo is None:
        verdict = "n/a"
    elif pbo < 0.3:
        verdict = "reasonable edge (PBO < 0.3)"
    elif pbo < 0.5:
        verdict = "borderline (0.3 ≤ PBO < 0.5)"
    elif pbo < 0.7:
        verdict = "coin flip (0.5 ≤ PBO < 0.7)"
    else:
        verdict = "likely overfit (PBO ≥ 0.7)"
    sharpe = m.get('sharpe', float('nan'))
    mdd = m.get('max_drawdown', float('nan'))
    cagr = m.get('cagr', float('nan'))
    parts.append(f"<p><span class='metric'><span class='k'>Sharpe</span><br>"
                 f"<span class='v'>{sharpe:.3f}</span></span>"
                 f"<span class='metric'><span class='k'>MaxDD</span><br>"
                 f"<span class='v'>{mdd:.2%}</span></span>"
                 f"<span class='metric'><span class='k'>CAGR</span><br>"
                 f"<span class='v'>{cagr:.2%}</span></span>"
                 f"<span class='metric'><span class='k'>Provider</span><br>"
                 f"<span class='v'>{esc(provider)}</span></span></p>")
    sweep_s = sweep.get('sharpe', float('nan'))
    sweep_s_s = f"{sweep_s:.3f}" if isinstance(sweep_s, (int, float)) else esc(sweep_s)
    parts.append(f"<p><strong>Sweep best:</strong> Sharpe {sweep_s_s} "
                 f"at params {esc(sweep.get('params', {}))}</p>")
    pbo_s = f"{pbo:.4f}" if isinstance(pbo, (int, float)) else esc(pbo)
    parts.append(f"<p><strong>PBO:</strong> {pbo_s} "
                 f"— <em>{esc(verdict)}</em></p>")
    gap = backtest.get("gap_vs_track_a")
    if gap:
        parts.append(f"<p class='handoff'><strong>Track A delta:</strong> {esc(gap)}</p>")
else:
    parts.append("<p class='skip'>backtest_result_free.pkl missing — section skipped.</p>")

parts.append("<div class='footer'>Generated by NB07 (Track B · free-only). "
             "Repo-relative paths only; no external CDNs; safe to view offline. "
             "Track A capstone lives at .notebook_state/portfolio_report.html — not modified here.</div>")
parts.append("</body></html>")

html_doc = "".join(parts)
OUT.write_text(html_doc, encoding="utf-8")
size = OUT.stat().st_size
print()
print(f"Wrote (repo-rel): .notebook_state/portfolio_report_free.html  ({size:,} bytes)")
print("Track A's portfolio_report.html NOT modified by this notebook.")



Artifact load summary:
  basket.json                      ok
  msft_free_evidence.pkl           ok
  xray_free.pkl                    ok
  smart_money_free.pkl             ok
  paper_blotter_free.pkl           ok
  backtest_result_free.pkl         ok

Wrote (repo-rel): .notebook_state/portfolio_report_free.html  (6,037 bytes)
Track A's portfolio_report.html NOT modified by this notebook.


## 3. Timed end-to-end run (optional) — how long does Track B take?

STORY_BIBLE Rule [#10](https://github.com/prajoria/OpenBB/issues/10): measure the real routine, don't hand-wave the tagline. Time the three heavy pieces of the Track B pipeline under free-only providers:

- **P2 evidence:** MSFT SEC filings + CBOE quote (small, fast).
- **P3 x-ray:** SEC N-PORT look-through for the ETF sleeve; the disk   cache at `.notebook_state/nport_cache/` (populated by NB03) makes   this near-instant on the warm path.
- **P6 backtest:** `buy_and_hold` over the 10-name basket via   `provider='cboe'`.

Expected wall-clock: **~15-25s total** on the warm path. Cold caches add ~30-60s. Either way, compute is not the constraint for Track B — the SEC rate-limit + CBOE EOD freshness window is.


In [ ]:
# [Track B / NB07 §3] Wall-clock the three heavy pieces under free-only providers
import asyncio, inspect, sys, threading, time
from datetime import date
from decimal import Decimal
import pathlib as _pl

sys.path.insert(0, str(_pl.Path("../portfolio").resolve()))

def _run_coro_sync(coro):
    result: dict = {}
    def _target():
        try:
            result["value"] = asyncio.run(coro)
        except BaseException as exc:  # noqa: BLE001
            result["error"] = exc
    t = threading.Thread(target=_target, daemon=True)
    t.start(); t.join()
    if "error" in result:
        raise result["error"]
    return result.get("value")

timings: list[tuple[str, float, str]] = []

def _timed(label: str, fn):
    t0 = time.perf_counter()
    try:
        rv = fn()
        if inspect.iscoroutine(rv):
            _run_coro_sync(rv)
        status = "ok"
    except Exception as exc:  # noqa: BLE001
        status = f"{type(exc).__name__}: {str(exc)[:80]}"
    dt = time.perf_counter() - t0
    timings.append((label, dt, status))
    print(f"  {label:<44} {dt:>7.2f}s   {status}")

print("Timed Monday-morning routine (Track B, free-only, this run):")
print()

def _step_p2_evidence():
    # Free P2 evidence: one CBOE quote for MSFT (cheap real call).
    from openbb import obb
    obb.equity.price.quote(symbol="MSFT", provider="cboe")

def _step_p3_xray():
    # SEC N-PORT look-through; disk cache populated by NB03 Track B
    # keeps this near-instant on the warm path.
    from _nport_lookthrough import nport_holdings, NportUnavailable
    for sym in ("QQQ", "VTI", "VNQ"):
        try:
            nport_holdings(sym)
        except NportUnavailable:
            pass

def _step_p6_backtest():
    from openbb import obb
    from openbb_backtest.models import (
        BacktestConfig, CommissionModel, SlippageModel, ComputeConfig,
    )
    universe = ["MSFT", "NVDA", "GOOGL", "AAPL", "AMD",
                "QQQ", "VTI", "VNQ", "BND", "GLD"]
    cfg = BacktestConfig(
        strategy="buy_and_hold",
        universe=universe,
        start=date(2023, 1, 3),
        end=date(2024, 12, 31),
        initial_cash=Decimal("100000"),
        commission=CommissionModel(kind="flat", value=Decimal("0"), min_per_trade=Decimal("0")),
        slippage=SlippageModel(kind="fixed_bps", value=Decimal("0")),
        compute=ComputeConfig(),
        benchmark="SPY",
        frequency="daily",
    )
    obb.backtest.run(cfg, strategy_params={"symbols": universe}, provider="cboe")

t_total = time.perf_counter()
_timed("P2 evidence: CBOE quote(MSFT)", _step_p2_evidence)
_timed("P3 x-ray:    SEC N-PORT look-through (warm cache)", _step_p3_xray)
_timed("P6 backtest: buy_and_hold 10-name (provider='cboe')", _step_p6_backtest)
total = time.perf_counter() - t_total

print()
print(f"  {'TOTAL (3 heavy pieces)':<44} {total:>7.2f}s")
print()
print(f"  vs. the '45-minute' tagline (2700s), that's {total / 2700 * 100:.2f}% of the budget.")
print("  Track B compute is not the constraint — SEC rate-limit + CBOE EOD freshness are.")


Timed Monday-morning routine (Track B, free-only, this run):



  P2 evidence: CBOE quote(MSFT)                   5.62s   ok
  P3 x-ray:    SEC N-PORT look-through (warm cache)    0.00s   ok


  P6 backtest: buy_and_hold 10-name (provider='cboe')    0.51s   ok

  TOTAL (3 heavy pieces)                          6.14s

  vs. the '45-minute' tagline (2700s), that's 0.23% of the budget.
  Track B compute is not the constraint — SEC rate-limit + CBOE EOD freshness are.


---

## What is NOT in this notebook

Same gaps as Track A NB07 plus the free-tier delta:

- **Analysis composite score.** The `Phase7Result` composite requires   `fmp_cached`. This notebook presents the P1-P6 free evidence bundle   in §2 and points at Track A NB02/NB07 for the score — same honest   hand-off NB02 (Track B) §7 already documented.
- **scrape-record as primary evidence.** Track A leans on scrape-  record snapshots for institutional-flow evidence; Track B uses SEC   N-PORT + SEC insiders instead, so §1 here is optional freshness   telemetry.
- **Total-return-adjusted backtest metrics.** CBOE is price-only. The   Sharpe / CAGR in §2's backtest section run ~1-2%/yr lower than   Track A on dividend-paying names. NB06 (Track B) §0.5 has the fine   print.
- **Intraday paper marks.** CBOE quotes are EOD-only; intraday marks   require Track A.
- **Analyst-rating capstone (NB08).** No free-authoritative source.   Track A covers it; Track B has no analog planned.

## 📚 Further reading

Every Investopedia link cited in Track A NB07 (backtesting, look-ahead bias, drawdown, Sharpe ratio, HHI, effective N, look-through, insider transactions, 13F, N-PORT) applies unchanged. Not re-cited here.

**Free-authoritative sources used across the Track B series:**

- **SEC EDGAR** — filings (Forms 10-K/Q, 3/4/5, 13F), N-PORT quarterly   fund holdings; primary regulatory source, no API key required,   subject to SEC's fair-access rate limit.
- **CBOE** — EOD price quotes + historicals; free-authoritative   price-only bars (no dividend reinvestment).
- **yfinance** — recorded snapshots for shape probing; NOT used as a   live path in production notebooks.

**Series pointers:**

- Track A capstone: [`../portfolio/07-offline-recording-and-end-to-end.ipynb`](../portfolio/07-offline-recording-and-end-to-end.ipynb)
- Series README: [`../portfolio/README.md`](../portfolio/README.md)
- STORY_BIBLE: [`../portfolio/STORY_BIBLE.md`](../portfolio/STORY_BIBLE.md)
